# OpenPlaque RCA Geometry Diagnostic — main branch baseline
This notebook does **not** implement a centerline. It determines how the solid `main`-branch RCA data are represented. Use **Runtime → Run all**. Google Drive is always mounted first.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Clone clean branch, then install ALL third-party dependencies before importing them.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK numpy matplotlib

import sys, os, time, shutil
from pathlib import Path
SRC=Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))

import numpy as np
import matplotlib.pyplot as plt
import pydicom
import SimpleITK as sitk
from openplaque.study import OpenPlaqueStudy
print('Dependencies loaded.')
print('Using clean OpenPlaque source:',SRC)


## 1. Stage and load RCA series 1035
The ZIP is copied from Drive to local Colab storage before extraction/scanning.

In [ ]:
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP=ROOT/'Full_DICOM.zip'
LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT_ROOT='/content/full_dicom_main_diag'
RCA_SERIES=1035
if not DRIVE_ZIP.exists(): raise FileNotFoundError(f'Missing {DRIVE_ZIP}')
t=time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying Full_DICOM.zip to local disk ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB)...',flush=True)
    shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f} s.',flush=True)
else:
    print('Local ZIP already staged.',flush=True)
shutil.rmtree(EXTRACT_ROOT,ignore_errors=True)
print('Extracting and scanning DICOM headers locally...',flush=True)
t=time.time(); study=OpenPlaqueStudy(str(LOCAL_ZIP),extract_root=EXTRACT_ROOT)
print(f'Found {len(study.series)} series in {time.time()-t:.1f} s.',flush=True)
match=[s for s in study.series if s['series_number']==RCA_SERIES]
if not match: raise ValueError(f'RCA series {RCA_SERIES} not found')
print('RCA inventory entry:',match[0])
ct_img,ct,rca_files=study.load_series(RCA_SERIES)
print('SimpleITK size xyz:',ct_img.GetSize())
print('NumPy shape zyx:',ct.shape)
print('Spacing xyz mm:',ct_img.GetSpacing())
print('Origin xyz:',ct_img.GetOrigin())
print('Direction:',ct_img.GetDirection())


## 2. Inspect DICOM geometry

In [ ]:
def getv(ds,name): return getattr(ds,name,None)
sample_idx=sorted(set([0,len(rca_files)//2,len(rca_files)-1]))
for i in sample_idx:
    ds=pydicom.dcmread(rca_files[i],stop_before_pixels=True,force=True)
    print('\n--- file',i,Path(rca_files[i]).name,'---')
    for tag in ['SeriesNumber','SeriesDescription','ImageType','InstanceNumber','Rows','Columns','PixelSpacing','SliceThickness','SpacingBetweenSlices','ImageOrientationPatient','ImagePositionPatient','SliceLocation','FrameOfReferenceUID']:
        print(f'{tag}:',getv(ds,tag))
positions=[]; instances=[]
for f in rca_files:
    ds=pydicom.dcmread(f,stop_before_pixels=True,force=True)
    if hasattr(ds,'ImagePositionPatient'):
        positions.append(np.asarray(ds.ImagePositionPatient,dtype=float)); instances.append(int(getattr(ds,'InstanceNumber',len(instances))))
if len(positions)>1:
    order=np.argsort(instances); P=np.asarray(positions)[order]
    steps=np.linalg.norm(np.diff(P,axis=0),axis=1)
    print('\nIPP step norm mm: median',float(np.median(steps)),'min',float(np.min(steps)),'max',float(np.max(steps)))
    print('First IPP:',P[0],'Last IPP:',P[-1],'straight displacement mm:',float(np.linalg.norm(P[-1]-P[0])))


## 3. Load the existing main-workflow RCA segmentation
Main convention: label 1 = vessel, label 2 = plaque.

In [ ]:
MASK_PATH=ROOT/'Segmentations'/'RCA_plaque_segmentation.nii.gz'
if not MASK_PATH.exists():
    hits=list(ROOT.rglob('RCA_plaque_segmentation.nii.gz'))
    if not hits: raise FileNotFoundError('Could not find RCA_plaque_segmentation.nii.gz under MyDrive/OpenPlaque')
    MASK_PATH=hits[0]
mask_img=sitk.ReadImage(str(MASK_PATH)); mask=sitk.GetArrayFromImage(mask_img)
print('Mask:',MASK_PATH)
print('Mask shape zyx:',mask.shape)
print('Mask spacing xyz:',mask_img.GetSpacing())
vals,counts=np.unique(mask,return_counts=True)
print('Label counts:',dict(zip(vals.tolist(),counts.tolist())))
if mask.shape!=ct.shape: raise ValueError(f'Mask/CT shape mismatch: {mask.shape} vs {ct.shape}')
vessel=mask==1; plaque=mask==2; foreground=vessel|plaque


## 4. Test the main-branch longitudinal-axis assumption

In [ ]:
slice_counts=foreground.sum(axis=(1,2)); active=np.where(slice_counts>0)[0]
if len(active)==0: raise ValueError('RCA segmentation has no foreground')
z0,z1=int(active[0]),int(active[-1]); sz=float(mask_img.GetSpacing()[2]); extent_mm=(z1-z0)*sz
print('Active slice range:',z0,'to',z1,'(',len(active),'active slices )')
print(f'Axis-0 foreground extent: {extent_mm:.1f} mm using z spacing {sz:.4f} mm')
print('Enough nominal extent for 10–50 mm interval:',extent_mm>=50.0)
cent_y=np.full(mask.shape[0],np.nan); cent_x=np.full(mask.shape[0],np.nan)
for z in active:
    yy,xx=np.where(foreground[z]); cent_y[z]=yy.mean(); cent_x[z]=xx.mean()
dist=(np.arange(mask.shape[0])-z0)*sz
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(dist[active],cent_x[active],label='centroid x'); ax.plot(dist[active],cent_y[active],label='centroid y'); ax.set_xlabel('Distance along array axis 0 (mm)'); ax.set_ylabel('Centroid (pixels)'); ax.set_title('RCA mask centroid vs stack position'); ax.legend(); ax.grid(True,alpha=.25); plt.show()
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(dist[active],slice_counts[active]); ax.set_xlabel('Distance along array axis 0 (mm)'); ax.set_ylabel('Foreground pixels/slice'); ax.set_title('RCA vessel+plaque area vs stack position'); ax.grid(True,alpha=.25); plt.show()


## 5. Representative vessel-centered slices

In [ ]:
zz,yy,xx=np.where(foreground); margin=20
ylo=max(0,int(yy.min())-margin); yhi=min(ct.shape[1],int(yy.max())+margin+1)
xlo=max(0,int(xx.min())-margin); xhi=min(ct.shape[2],int(xx.max())+margin+1)
zs=np.linspace(z0,z1,9).round().astype(int)
fig,axes=plt.subplots(3,3,figsize=(12,12))
for ax,z in zip(axes.ravel(),zs):
    ax.imshow(ct[z,ylo:yhi,xlo:xhi],cmap='gray',vmin=-200,vmax=800)
    v=vessel[z,ylo:yhi,xlo:xhi]; p=plaque[z,ylo:yhi,xlo:xhi]
    if np.any(v): ax.contour(v.astype(float),levels=[0.5],linewidths=1.0)
    if np.any(p): ax.contour(p.astype(float),levels=[0.5],linewidths=1.5)
    ax.set_title(f'z={z}; d={(z-z0)*sz:.1f} mm; n={int(slice_counts[z])}'); ax.axis('off')
plt.tight_layout(); plt.show()
print('DIAGNOSTIC COMPLETE.')
